# Production calibrators
Train and evaluate production-ready calibrators (affine + vector scaling) on UXM deconvolution predictions.

In [30]:
import json
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator
from methyldl.deconvolution.vector_scaling_calibrator import VectorScalingCalibrator, VectorScalingCalibratorCV

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data loading

In [19]:
with open("../App/labels_dict.json", "r") as f:
    labels_to_ctype_names = json.load(f)
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())
print(f"{len(ctype_names_list)} cell types: {ctype_names_list[:5]} ...")

39 cell types: ['Adipocytes', 'Bladder-Ep', 'Blood-B', 'Blood-Granul', 'Blood-Mono+Macro'] ...


In [20]:
uxm_pred_folder = Path("../Data/training_data/mixture_predictions/uxm")

target_prop = np.load(uxm_pred_folder / "target_proportions.npz")["arr_0"]
uxm_train_data = np.load(uxm_pred_folder / "uxm_results_train.npz")["arr_0"]
uxm_val_data = np.load(uxm_pred_folder / "uxm_results_valid.npz")["arr_0"]
uxm_test_data = np.load(uxm_pred_folder / "uxm_results_test.npz")["arr_0"]

print(f"Train: {uxm_train_data.shape}, Val: {uxm_val_data.shape}, Test: {uxm_test_data.shape}")
print(f"Targets: {target_prop.shape}")

Train: (100000, 39), Val: (100000, 39), Test: (100000, 39)
Targets: (100000, 39)


## Evaluation helper

In [21]:
def evaluate_test(test_pred: dict, round_: int = 6) -> pd.DataFrame:
    """Compute deconvolution metrics for each method and return a sorted DataFrame."""
    results = {
        name: compute_deconvolution_metrics(
            pred=pred, target=target_prop, class_names=ctype_names_list
        )
        for name, pred in test_pred.items()
    }
    # Drop per-class arrays for compact display
    results = {
        k: {m: v for m, v in metrics.items() if "per_class" not in m}
        for k, metrics in results.items()
    }
    df = pd.DataFrame(results).T
    float_cols = [
        "mae", "mse", "kl", "max_error", "cosine_sim",
        "loa_lower", "loa_upper", "loa_width",
        "worst_class_loa_lower", "worst_class_loa_upper", "worst_class_loa_width",
    ]
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype(float).round(round_)
    return df.sort_values("mse")

## 1. Affine (Linear) calibrator

In [22]:
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_val_data, target_prop)

# Predict with both normalization strategies
uxm_test_lin_clip_norm, _ = linear_cal.predict(uxm_test_data, norm_method="clip01-normalize")
uxm_test_lin_simplex, _ = linear_cal.predict(uxm_test_data, norm_method="simplex-projection")

## 2. Vector Scaling calibrator — grid search

In [24]:
param_grid = {
    "reg_lambda": [0.0, 1e-4, 1e-3, 1e-2, 1e-1],
    "lr": [1e-2],
    "max_iter": [500],
    "scheduler": ["cosine"],
    "batch_size": [None],  # full-batch
    "patience": [50],
    "tol": [1e-7],
}

def prod_len(d: dict) -> int:
    return reduce(lambda x, y: x * y, (len(v) for v in d.values()), 1)

results_gs = []
best_val_loss = np.inf
best_vs_model: VectorScalingCalibrator | None = None

n_combos = prod_len(param_grid)
with tqdm(total=n_combos, desc="VS grid search") as pbar:
    for reg_lambda in param_grid["reg_lambda"]:
        for lr in param_grid["lr"]:
            for max_iter in param_grid["max_iter"]:
                for scheduler in param_grid["scheduler"]:
                    for batch_size in param_grid["batch_size"]:
                        for patience in param_grid["patience"]:
                            for tol in param_grid["tol"]:
                                vs_cal = VectorScalingCalibrator(
                                    reg_lambda=reg_lambda,
                                    optimizer="adam",
                                    lr=lr,
                                    scheduler=scheduler,
                                    max_iter=max_iter,
                                    batch_size=batch_size,
                                    patience=patience,
                                    tol=tol,
                                    device="cuda",
                                    verbose=False,
                                )
                                vs_cal.fit(
                                    X=uxm_train_data,
                                    y=target_prop,
                                    X_val=uxm_val_data,
                                    y_val=target_prop,
                                )
                                results_gs.append({
                                    "reg_lambda": reg_lambda,
                                    "lr": lr,
                                    "max_iter": max_iter,
                                    "scheduler": scheduler,
                                    "batch_size": batch_size,
                                    "patience": patience,
                                    "tol": tol,
                                    "best_train_loss": vs_cal.best_metrics_["train_loss"],
                                    "best_val_loss": vs_cal.best_metrics_["val_loss"],
                                    "best_train_mse": vs_cal.best_metrics_["train_mse"],
                                    "best_val_mse": vs_cal.best_metrics_["val_mse"],
                                    "best_epoch": vs_cal.best_epoch_,
                                })
                                if vs_cal.best_metrics_["val_loss"] < best_val_loss:
                                    
                                    best_val_loss = vs_cal.best_metrics_["val_loss"]
                                    best_vs_model = vs_cal
                                pbar.update(1)

gs_df = pd.DataFrame(results_gs).sort_values("best_val_loss")
gs_df

VS grid search:   0%|          | 0/5 [00:00<?, ?it/s]

,reg_lambda,lr,max_iter,scheduler,batch_size,patience,tol,best_train_loss,best_val_loss,best_train_mse,best_val_mse,best_epoch
0,0.0000,0.01,500,cosine,None,50,1.000000e-07,1.419350,1.536281,0.000087,0.000269,54
1,0.0001,0.01,500,cosine,None,50,1.000000e-07,1.419720,1.536653,0.000085,0.000266,54
2,0.0010,0.01,500,cosine,None,50,1.000000e-07,1.422419,1.539670,0.000075,0.000251,52
3,0.0100,0.01,500,cosine,None,50,1.000000e-07,1.432609,1.553051,0.000062,0.000246,21
4,0.1000,0.01,500,cosine,None,50,1.000000e-07,1.441427,1.564164,0.000083,0.000282,22


In [38]:
# actual 5fold grid search with VectorScalingCalibratorCV
vs_cal_cv = VectorScalingCalibratorCV(
    reg_lambda_list=[0.0, 1e-4],
    lr_list=[1e-3, 1e-2],
    max_iter_list=[250, 500],
    optimizer="adam",
    scheduler="cosine",
    patience=50,
    tol=1e-7,
    n_folds=3,
    verbose=True,
)
vs_cal_cv.fit(
    X_train=uxm_train_data,
    y_train=target_prop,
    X_val=uxm_val_data,
    y_val=target_prop,
)

CV grid search: 100%|██████████| 24/24 [06:57<00:00, 17.38s/it, best_val_loss=1.4730566e+00, best_params={'batch_size': 50000, 'lr': 0.01, 'max_iter': 500, 'optimizer': 'adam', 'patience': 50, 'reg_lambda': 0.0, 'scheduler': 'cosine', 'tol': 1e-07}] 


Epoch -1: train_loss = 1.5053788e+00, train_mse = 1.9922710e-04


ZeroDivisionError: integer modulo by zero

## 3. Test-set evaluation

In [28]:
uxm_test_vs = best_vs_model.predict_proba(uxm_test_data)

results_df = evaluate_test({
    "UXM (no calibration)": uxm_test_data,
    "Affine + clip01-normalize": uxm_test_lin_clip_norm,
    "Affine + simplex-projection": uxm_test_lin_simplex,
    "Vector Scaling (best)": uxm_test_vs,
})
results_df.sort_values("mse")

,mae,mse,kl,max_error,cosine_sim,loa_lower,loa_upper,loa_width,worst_class_idx,worst_class_name,worst_class_loa_lower,worst_class_loa_upper,worst_class_loa_width
Affine + simplex-projection,0.004211,0.000208,0.124543,0.363513,0.986401,-0.028240,0.028240,0.056480,11,Colon-Fibro,-0.063390,0.085064,0.148454
Vector Scaling (best),0.004840,0.000222,0.111505,0.369014,0.984868,-0.029217,0.029217,0.058434,11,Colon-Fibro,-0.059224,0.072366,0.131590
Affine + clip01-normalize,0.005033,0.000255,0.133534,0.411706,0.985778,-0.031280,0.031280,0.062561,11,Colon-Fibro,-0.063627,0.085552,0.149179
UXM (no calibration),0.005563,0.000294,0.145682,0.513800,0.985040,-0.033601,0.033601,0.067202,11,Colon-Fibro,-0.068714,0.095688,0.164402
